# Finetuning de Stable Diffusion v1-4 con oldbookillustrations (CPU)

Notebook optimizado para evitar OOM usando streaming, batch pequeño, acumulación de gradiente y gradient checkpointing.

In [1]:
!pip -q install --upgrade diffusers transformers accelerate datasets safetensors torchvision huggingface_hub peft


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, io, gc, math, random
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from torchvision import transforms
from PIL import Image

from diffusers import StableDiffusionPipeline, DDPMScheduler
from huggingface_hub import login

/usr/local/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/usr/local/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


## Parámetros

Configura el token de Hugging Face con la variable de entorno `HF_TOKEN` (recomendado) o ejecútalo con `login()` cuando se pida.

In [3]:
MODEL_ID = "CompVis/stable-diffusion-v1-4"
DATASET_ID = "gigant/oldbookillustrations"

OUTPUT_DIR = "sd14-oldbook-lora"
REPO_ID = "pedrojrg/gen_ai_images"

SEED = 42

# Baja a 384 si vas justo de RAM/CPU. 512 en CPU suele ser inviable.
RESOLUTION = 384

EPOCHS = 2
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

LR = 1e-4  # LoRA suele ir mejor algo más alto que full finetune

# Streaming: no hay len(), así que fijamos pasos por epoch
STEPS_PER_EPOCH = 80
SAMPLES_PER_EPOCH = STEPS_PER_EPOCH * TRAIN_BATCH_SIZE

# Shuffle buffer controla RAM (más = mejor mezcla, más RAM)
SHUFFLE_BUFFER = 200

# LoRA
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

SAVE_EVERY_N_STEPS = 0  # 0 para desactivar checkpoints intermedios

PROMPT_TEST = "a detailed black and white engraving of a medieval city, high detail, old book illustration"


In [4]:
random.seed(SEED)
torch.manual_seed(SEED)

# Limitar threads para no saturar CPUs pequeñas / compartidas
_cpu = os.cpu_count() or 1
torch.set_num_threads(min(2, _cpu))
torch.set_num_interop_threads(1)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

# En CPU, bfloat16 reduce memoria de activaciones en muchos casos
USE_BF16 = (device.type == "cpu")
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("Autocast enabled:", USE_BF16, "dtype:", AMP_DTYPE)

Device: cpu
Autocast enabled: True dtype: torch.bfloat16


## Login (opcional)

Si `HF_TOKEN` está definido como variable de entorno, no necesitas pegar nada aquí.

In [5]:
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    login(token=hf_token)
    print("HF login OK (env token).")
else:
    print("HF_TOKEN no está definido. Si necesitas autenticar, ejecuta login() manualmente.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK (env token).


## Generar imagen ANTES del finetuning

In [ ]:
pipe_before = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    safety_checker=None
).to(device)

# Reduce uso de memoria en inferencia
pipe_before.enable_attention_slicing()
pipe_before.enable_vae_slicing()

gen = torch.Generator(device=device).manual_seed(SEED)

with torch.inference_mode():
    img_before = pipe_before(
        prompt=PROMPT_TEST,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=gen
    ).images[0]

os.makedirs("outputs", exist_ok=True)
before_path = os.path.join("outputs", "before_finetune.png")
img_before.save(before_path)
before_path

In [ ]:
# Libera memoria del pipeline de inferencia antes de entrenar
del pipe_before
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
print("Pipeline 'before' liberado.")

## Preparar modelo para entrenamiento (UNet)

Se congela VAE y text encoder para reducir coste y memoria.

In [6]:
import gc
import torch
from diffusers import StableDiffusionPipeline, DDPMScheduler

# Necesario para LoRA en diffusers
try:
    from peft import LoraConfig
except Exception as e:
    raise RuntimeError(
        "Falta 'peft'. Instálalo en tu entorno antes de entrenar: pip install peft"
    ) from e

# En CPU, bf16 reduce RAM real si casteas explícitamente el modelo
# (autocast NO reduce el tamaño de parámetros). :contentReference[oaicite:3]{index=3}
BASE_DTYPE = torch.bfloat16 if (device.type == "cpu") else torch.float16

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=BASE_DTYPE,
    safety_checker=None,
)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
vae = pipe.vae
unet = pipe.unet

noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Congelar TODO lo base
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# Activar gradient checkpointing (reduce activaciones; no arregla lo de AdamW full, pero ayuda)
unet.enable_gradient_checkpointing()

# Añadir LoRA al UNet: ahora solo esas capas serán entrenables (muy poca RAM vs full). :contentReference[oaicite:4]{index=4}
unet_lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],  # estándar atención SD1.x
)
unet.add_adapter(unet_lora_config)

# Asegurar que SOLO LoRA es trainable y en float32 (más estable)
lora_params = []
for n, p in unet.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
        lora_params.append(p)

unet.train()
vae.eval()
text_encoder.eval()

# Mover a dispositivo
vae.to(device, dtype=BASE_DTYPE)
text_encoder.to(device, dtype=BASE_DTYPE)
unet.to(device, dtype=BASE_DTYPE)

# Liberar contenedor
del pipe
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

trainable = sum(p.numel() for p in lora_params)
total = sum(p.numel() for p in unet.parameters())
print(f"UNet total params: {total:,} | trainable (LoRA): {trainable:,}")
print("Componentes listos.")


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its r

UNet total params: 861,115,332 | trainable (LoRA): 1,594,368
Componentes listos.


## Dataset en streaming + DataLoader

En streaming, las imágenes pueden llegar como `dict` con `bytes` o `path`.

In [7]:
from datasets import load_dataset
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

def has_caption(ex):
    cap = ex.get("info_alt", None)
    return cap is not None and isinstance(cap, str) and len(cap.strip()) > 0

ds = load_dataset(DATASET_ID, split="train", streaming=True).filter(has_caption)

# Shuffle aproximado con buffer limitado (RAM-safe)
ds = ds.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED)

print(ds)


IterableDataset({
    features: ['rawscan', '1600px', 'info_url', 'info_src', 'info_alt', 'artist_name', 'artist_birth_date', 'artist_death_date', 'artist_countries', 'book_title', 'book_authors', 'book_publishers', 'date_published', 'openlibrary-url', 'tags', 'illustration_source_name', 'illustration_source_url', 'illustration_subject', 'illustration_format', 'engravers', 'image_title', 'image_caption', 'image_description', 'rawscan_url', '1600px_url'],
    num_shards: 13
})


In [8]:
import io
import torch
from dataclasses import dataclass
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Wrapper para que PyTorch NO intente hacer len() (evita SequentialSampler).
class TorchHFIterableDataset(IterableDataset):
    def __init__(self, hf_iterable):
        super().__init__()
        self.hf_iterable = hf_iterable

    def __iter__(self):
        yield from self.hf_iterable

# Tomamos un número fijo de muestras por epoch (streaming)
ds_epoch = TorchHFIterableDataset(ds.take(SAMPLES_PER_EPOCH))

image_transforms = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB") if isinstance(im, Image.Image) else im),
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def tokenize_caption(caption: str):
    tok = tokenizer(
        caption,
        truncation=True,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    )
    return tok.input_ids[0]

def load_pil_image(img_obj):
    if isinstance(img_obj, Image.Image):
        return img_obj

    if isinstance(img_obj, dict):
        b = img_obj.get("bytes", None)
        p = img_obj.get("path", None)

        if b is not None:
            return Image.open(io.BytesIO(b)).convert("RGB")
        if p is not None and isinstance(p, str) and len(p) > 0:
            return Image.open(p).convert("RGB")

        raise TypeError(f"Image dict sin 'bytes' ni 'path' útil. Keys={list(img_obj.keys())}")

    raise TypeError(f"Unexpected image type: {type(img_obj)}")

@dataclass
class Batch:
    pixel_values: torch.Tensor
    input_ids: torch.Tensor

def collate_fn(examples):
    pixel_values = []
    input_ids = []

    for ex in examples:
        img_raw = ex["1600px"]
        cap = ex["info_alt"]

        img = load_pil_image(img_raw)
        pixel_values.append(image_transforms(img))
        input_ids.append(tokenize_caption(cap))

    return Batch(
        pixel_values=torch.stack(pixel_values),
        input_ids=torch.stack(input_ids),
    )

train_loader = DataLoader(
    ds_epoch,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=0,      # CPU baja: 0 evita duplicar RAM por workers
    collate_fn=collate_fn,
    pin_memory=False,
    persistent_workers=False,
)

batch = next(iter(train_loader))
print("Batch pixel_values:", batch.pixel_values.shape, batch.pixel_values.dtype)
print("Batch input_ids:", batch.input_ids.shape, batch.input_ids.dtype)


Batch pixel_values: torch.Size([1, 3, 384, 384]) torch.float32
Batch input_ids: torch.Size([1, 77]) torch.int64


## Entrenamiento

Incluye autocast bfloat16 en CPU, acumulación de gradiente y limitación de batches por epoch para evitar OOM.

In [ ]:
import gc
import torch.nn.functional as F

trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR)

global_step = 0

def make_epoch_loader(epoch_idx: int):
    shuffled = ds.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED + epoch_idx)
    epoch_ds = TorchHFIterableDataset(shuffled.take(SAMPLES_PER_EPOCH))
    return DataLoader(
        epoch_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
        pin_memory=False,
        persistent_workers=False,
    )

vae_dtype = next(vae.parameters()).dtype   # ← bfloat16
unet_dtype = next(unet.parameters()).dtype # ← bfloat16

for epoch in range(EPOCHS):
    train_loader = make_epoch_loader(epoch)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        # --- mover a device y CASTEAR AL DTYPE DEL VAE ---
        pixel_values = batch.pixel_values.to(device=device, dtype=vae_dtype)
        input_ids = batch.input_ids.to(device)

        # --- VAE + Text encoder (congelados) ---
        with torch.inference_mode():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * 0.18215
            encoder_hidden_states = text_encoder(input_ids)[0]

        # --- ruido ---
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # --- UNet (LoRA trainable) ---
        with torch.autocast(
            device_type=device.type,
            dtype=unet_dtype,
            enabled=(device.type == "cpu")
        ):
            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")

        (loss / GRAD_ACCUM_STEPS).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % 10 == 0:
                print(f"epoch={epoch} opt_step={global_step} loss={loss.item():.4f}")

        # --- limpieza agresiva de RAM ---
        del pixel_values, input_ids, latents, encoder_hidden_states
        del noise, noisy_latents, noise_pred, loss

        if (step + 1) % 20 == 0:
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

print("Training finished. optimizer steps:", global_step)

## Guardar modelo finetuneado

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

pipe_finetuned = StableDiffusionPipeline(
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    unet=unet,
    scheduler=noise_scheduler,
    safety_checker=None,
    feature_extractor=None
).to(device)

# Optimizaciones de memoria para inferencia
pipe_finetuned.enable_attention_slicing()
pipe_finetuned.enable_vae_slicing()

pipe_finetuned.save_pretrained(OUTPUT_DIR)
OUTPUT_DIR

## Subida a Hugging Face (opcional)

In [ ]:
if REPO_ID is None:
    print("REPO_ID no definido. Si quieres subir el modelo, asigna REPO_ID y vuelve a ejecutar.")
else:
    pipe_finetuned.push_to_hub(REPO_ID)
    print("Uploaded to:", f"https://huggingface.co/{REPO_ID}")

## Generar imagen DESPUÉS del finetuning

In [ ]:
gen = torch.Generator(device=device).manual_seed(SEED)

with torch.inference_mode():
    img_after = pipe_finetuned(
        prompt=PROMPT_TEST,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=gen
    ).images[0]

after_path = os.path.join("outputs", "after_finetune.png")
img_after.save(after_path)
after_path

## Guardar link del repo en .txt (si aplica)

In [ ]:
if REPO_ID is None:
    print("REPO_ID no definido; no se crea el txt.")
else:
    link = f"https://huggingface.co/{REPO_ID}"
    txt_path = os.path.join("outputs", "huggingface_repo_link.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(link + "\n")
    txt_path